In [2]:
import pandas as pd

In [3]:
# Load dengue csv file
DENGUE_CSV = (
    "https://raw.githubusercontent.com/"
    "CodexonB21/Data_Scraping/main/output/dengue_cases.csv"
)
try:
    # Read the CSV file
    dengue = pd.read_csv(DENGUE_CSV)
except Exception as exc:
    # Display error message if the data set cannot found
    raise RuntimeError(
        "Could not download the CSV. Download output/dengue_cases.csv "
        "from the source repository and set DENGUE_CSV to its local path."
    ) from exc

# Display the number of rows and columns
print("Shape:", dengue.shape)

# Display the names of all columns
print("Columns:", dengue.columns.tolist())

# Display the first five rows
display(dengue.head())

Shape: (26338, 7)
Columns: ['District', 'Number_of_Cases', 'Week_Start_Date', 'Month', 'Year', 'Week', 'Week_End_Date']


,District,Number_of_Cases,Week_Start_Date,Month,Year,Week,Week_End_Date
0,Ampara,0,12/23/2006,12,2006,52,12/29/2006
1,Anuradhapura,4,12/23/2006,12,2006,52,12/29/2006
2,Badulla,2,12/23/2006,12,2006,52,12/29/2006
3,Batticaloa,0,12/23/2006,12,2006,52,12/29/2006
4,Colombo,71,12/23/2006,12,2006,52,12/29/2006


In [4]:
# Clean and validate dataset

# Define the columns required for the analysis
required = {"District", "Number_of_Cases", "Week_Start_Date"}

# Identify the missing column
missing_required = required - set(dengue.columns)

# Stop execution if any required columns are missing
if missing_required:
    raise ValueError(f"Missing required columns: {missing_required}")

# Convert the week start date column to datetime format
# Invalid date values are converted to NaT
dengue["Week_Start_Date"] = pd.to_datetime(
    dengue["Week_Start_Date"], errors="coerce")

# Convert dengue case counts to numeric values
# Invalid numeric values are converted to NaN
dengue["Number_of_Cases"] = pd.to_numeric(
    dengue["Number_of_Cases"], errors="coerce")

# Remove records with missing district names, dates, or case counts
# .copy() creates an independent DataFrame for further processing
dengue = dengue.dropna(
    subset=["District", "Week_Start_Date", "Number_of_Cases"]
).copy()

# Remove records containing negative dengue case counts
dengue = dengue[dengue["Number_of_Cases"] >= 0].copy()

# Convert district names to strings and remove leading/trailing spaces
dengue["District"] = dengue["District"].astype(str).str.strip()

# Identify duplicate records for the same district and week
# Keep the last record for each district-week combination
dengue = dengue.drop_duplicates(
    subset=["District", "Week_Start_Date"], keep="last")

# Sort records by district and date, then reset the row index
dengue = dengue.sort_values(["District", "Week_Start_Date"]).reset_index(drop=True)

# Display the number of rows and columns after cleaning
print("Cleaned shape:", dengue.shape)

# Display the earliest and latest dates
print("Date range:", dengue["Week_Start_Date"].min(), "to", dengue["Week_Start_Date"].max())

# Count the number of unique districts and display
print("Number of districts:", dengue["District"].nunique())
print("Districts : ", dengue["District"].unique())

# Check for remaining missing values in each column
print("Missing values:\n", dengue.isna().sum())

# Display descriptive statistics for all columns
display(dengue.describe(include="all").T)


Cleaned shape: (25818, 7)
Date range: 2006-12-23 00:00:00 to 2026-06-13 00:00:00
Number of districts: 27
Districts :  ['Ampara' 'Anuradhapura' 'Badulla' 'Batticaloa' 'Colombo' 'Galle'
 'Gampaha' 'Hambantota' 'Jaffna' 'Kalmunai' 'Kalutara' 'Kandy' 'Kegalle'
 'Kilinochchi' 'Kurunegala' 'Mannar' 'Matale' 'Matara' 'Monaragala'
 'Mullaitivu' 'Nuwara Eliya' 'Polonnaruwa' 'Puttalam' 'Puttlam'
 'Ratnapura' 'Trincomalee' 'Vavuniya']
Missing values:
 District           0
Number_of_Cases    0
Week_Start_Date    0
Month              0
Year               0
Week               0
Week_End_Date      0
dtype: int64


,count,unique,top,freq,mean,min,25%,50%,75%,max,std
District,25818,27,Ampara,993,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Number_of_Cases,25818.0,NaN,NaN,NaN,32.595089,0.0,2.0,9.0,30.0,2631.0,81.974166
Week_Start_Date,25818,NaN,NaN,NaN,2016-09-17 04:03:37.522658560,2006-12-23 00:00:00,2011-10-29 00:00:00,2016-10-01 00:00:00,2021-07-31 00:00:00,2026-06-13 00:00:00,NaN
Month,25818.0,NaN,NaN,NaN,6.416918,1.0,3.0,6.0,9.0,12.0,3.461463
Year,25818.0,NaN,NaN,NaN,2016.236657,2006.0,2011.0,2016.0,2021.0,2026.0,5.644433
Week,25818.0,NaN,NaN,NaN,26.214502,1.0,13.0,26.0,39.0,53.0,15.073972
Week_End_Date,25818,993,6/19/2026,26,NaN,NaN,NaN,NaN,NaN,NaN,NaN
